## Plotly

In [6]:
import pandas as pd
import sqlite3
import plotly.graph_objects as go
import numpy as np

## 1 Установливаем соединение с базой данных.

In [7]:
conn = sqlite3.connect('/Users/cleganeb/Desktop/DSB9_Pandas_SQL_Data_Visual.ID_1577652-1/src/data/checking-logs.sqlite')

## 2 Достанем и подготовим данные из таблицы checker (только пользователи).

In [8]:
query = """
SELECT 
    uid,
    timestamp,
    numTrials
FROM checker
WHERE uid LIKE 'user_%' AND labname = 'project1' AND status = 'ready'
"""
checker = pd.io.sql.read_sql(query, conn)
checker["timestamp"] = pd.to_datetime(checker["timestamp"])
checker["date"] = checker["timestamp"].dt.date

daily = (checker.groupby(["date", "uid"])["numTrials"].max().unstack().sort_index())

daily = daily.ffill().fillna(0)

num_days = daily.shape[0]
y_max = float(daily.max().max())

print("days:", num_days, "| users:", daily.shape[1], "| y_max:", y_max)
display(daily)

days: 19 | users: 27 | y_max: 164.0


uid,user_1,user_10,user_11,user_12,user_13,user_14,user_15,user_16,user_17,user_18,...,user_26,user_27,user_28,user_29,user_3,user_30,user_31,user_4,user_6,user_8
date,,,,,,,,,,,,,,,,,,,,,
2020-04-17,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,0.0,0.0
2020-04-18,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,0.0,0.0
2020-04-19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.0,0.0,0.0
2020-04-22,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,11.0,0.0,0.0
2020-04-23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,2.0,0.0,20.0,0.0,0.0
2020-04-24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,2.0,0.0,27.0,0.0,0.0
2020-05-03,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,4.0,0.0,35.0,0.0,0.0
2020-05-04,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,5.0,0.0,48.0,0.0,0.0
2020-05-05,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,5.0,0.0,53.0,0.0,0.0


## 3 Построим график.

In [9]:
xaxis_max = 20  
xaxis_range = [0, xaxis_max]

x_init = np.array([1])

initial_data = []
for uid in daily.columns:
    initial_data.append(
        go.Scatter(
            x=x_init,
            y=[daily.iloc[0][uid]],
            mode="lines+markers",
            name=uid,
        )
    )

frames = []
for f in range(1, num_days + 1):
    x_axis = np.arange(1, f + 1)
    curr = []
    for uid in daily.columns:
        y_axis = daily.iloc[:f][uid].to_numpy()
        curr.append(
            go.Scatter(
                x=x_axis,
                y=y_axis,
                mode="lines+markers",
                name=uid,
            )
        )
    frames.append(go.Frame(data=curr, name=str(f)))

fig = go.Figure(
    data=initial_data,
    layout={
        "title": {"text": "Dynamic of commits per user in project1"},
        "template": "plotly", 
        "xaxis": {
            "range": xaxis_range,
            "tickmode": "linear",
            "tick0": 0,
            "dtick": 2,
            "showgrid": True,
            "title": "",
        },
        "yaxis": {
            "range": [0, 170],
            "dtick": 20,
            "showgrid": True,
            "title": "",
        },
        "legend": {"title": None, "font": dict(size=12), "tracegroupgap": 0},
        "updatemenus": [
            {
                "type": "buttons",
                "showactive": False,
                "x": -0.06, 
                "y": 0.88, 
                "xanchor": "right", 
                "yanchor": "bottom", 
                "buttons":[
                    {
                        "method": "animate",
                        "label": "play",
                        "args": [None, {
                            "frame": {"duration": 200, "redraw": True},
                            "fromcurrent": True,
                            "transition": {"duration": 0}
                        }],
                    }
                ],
            }
        ],
    },
    frames=frames,
)

fig.show()

## 4 Закроем соединение с базой данных.

In [10]:
conn.close()